# KPIs Operacionais — OMBUDS

Notebook exploratório para validar métricas base antes de virar view materializada no Supabase (Fase 1 do TDD `2026-04-11-analytics-ml-foundation-tdd.md`).

**Como usar**: clica no primeiro bloco de código abaixo e aperta `Shift+Enter`. Continua até o fim. Se mudar algo num gráfico, só precisa re-rodar o bloco daquele gráfico.

**Pré-requisito**: ambiente Python criado conforme `notebooks/README.md` e `.env.local` na raiz do projeto com `DATABASE_URL`.

## 1. Setup — conectar no Supabase

Carrega as variáveis de ambiente do projeto e abre conexão read-only com o banco. Roda uma vez.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Carrega .env.local da raiz do projeto (um nível acima de notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env.local")

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL não encontrada — verifique .env.local na raiz do projeto")

# SQLAlchemy precisa do driver explícito no scheme
if DATABASE_URL.startswith("postgresql://"):
    DATABASE_URL = DATABASE_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

# Teste rápido
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), now() as agora")).fetchone()
print(f"✓ Conectado em '{result[0]}' às {result[1]}")

## 2. Carregar demandas num DataFrame

Uma query só que traz tudo que precisamos: demanda + processo (pra ter a atribuição) + assistido (pra ter status prisional) + defensor (pra agrupar por pessoa). Filtra só ativas (não deletadas).

Depois de rodar esse bloco, `demandas` fica na memória e todos os próximos blocos usam ela sem re-consultar o banco.

In [ ]:
query = """
SELECT
    d.id,
    d.ato,
    d.status,
    d.prazo,
    d.data_entrada,
    d.created_at,
    d.defensor_id,
    d.processo_id,
    d.prioridade,
    d.reu_preso,
    p.atribuicao,
    p.numero_autos,
    a.nome AS assistido_nome,
    a.status_prisional,
    u.email AS defensor_email
FROM demandas d
LEFT JOIN processos p ON p.id = d.processo_id
LEFT JOIN assistidos a ON a.id = d.assistido_id
LEFT JOIN users u ON u.id = d.defensor_id
WHERE d.deleted_at IS NULL
"""

demandas = pd.read_sql(
    query,
    engine,
    parse_dates=["prazo", "data_entrada", "created_at"],
)

# Label amigável para defensor (nome do email antes do @)
demandas["defensor_nome"] = demandas["defensor_email"].fillna("(sem defensor)").str.split("@").str[0]

# Label amigável para atribuição (JURI_CAMACARI -> Júri, etc.)
ATRIB_LABEL = {
    "JURI_CAMACARI": "Júri",
    "VVD_CAMACARI": "VVD",
    "EXECUCAO_PENAL": "Exec. Penal",
    "SUBSTITUICAO": "Substituição",
    "SUBSTITUICAO_CIVEL": "Subst. Cível",
    "GRUPO_JURI": "Grupo Júri",
}
demandas["atribuicao_label"] = demandas["atribuicao"].map(ATRIB_LABEL).fillna("(sem processo)")

print(f"✓ {len(demandas)} demandas ativas carregadas")
demandas.head()

### Visão geral rápida

In [ ]:
print(f"Total de demandas ativas: {len(demandas)}")
print(f"Defensores com demandas:   {demandas['defensor_id'].nunique()}")
print(f"Atribuições distintas:     {demandas['atribuicao_label'].nunique()}")
print(f"Assistidos distintos:      {demandas['assistido_nome'].nunique()}")
print(f"Período (criação):         {demandas['created_at'].min().date()} → {demandas['created_at'].max().date()}")
print()
print("Por atribuição:")
print(demandas['atribuicao_label'].value_counts().to_string())

## 3. KPI 1 — Throughput por semana

Quantas demandas são **criadas** a cada semana. Ajuda a ver sazonalidade e sobrecarga.

> **Pra ler**: picos = entrada de lote (importação PJe, pauta nova). Vales = férias, recessos. Tendência crescente = volume geral subindo.

In [ ]:
# Agrupa por semana
weekly = (
    demandas.dropna(subset=["created_at"])
    .groupby(pd.Grouper(key="created_at", freq="W-MON"))
    .size()
    .reset_index(name="qtd")
)

fig = px.bar(
    weekly,
    x="created_at",
    y="qtd",
    title="Demandas criadas por semana",
    labels={"created_at": "Semana", "qtd": "Quantidade"},
)
fig.update_layout(showlegend=False, bargap=0.1)
fig.show()

## 4. KPI 2 — Backlog por atribuição × status

Onde o volume está parado. Cada barra é uma atribuição; as cores mostram em que status as demandas estão empilhadas.

> **Pra ler**: se uma barra está dominada por `5_TRIAGEM`, você tem gargalo de triagem. Se está dominada por `2_ATENDER`, gargalo na execução. `CONCLUIDO` grande é histórico saudável.

In [ ]:
backlog = (
    demandas.groupby(["atribuicao_label", "status"])
    .size()
    .reset_index(name="total")
    .sort_values("total", ascending=False)
)

fig = px.bar(
    backlog,
    x="atribuicao_label",
    y="total",
    color="status",
    title="Backlog por Atribuição × Status",
    labels={"atribuicao_label": "Atribuição", "total": "Quantidade", "status": "Status"},
    barmode="stack",
)
fig.show()

# Tabela cruzada pra olhar número preciso
demandas.pivot_table(
    index="atribuicao_label",
    columns="status",
    values="id",
    aggfunc="count",
    fill_value=0,
)

## 5. KPI 3 — Distribuição de dias até o prazo

Quantas demandas estão a quantos dias de vencer. A linha vermelha vertical é o **hoje**.

> **Pra ler**: tudo à esquerda do zero já estourou o prazo. Cluster entre 0 e 7 dias é a zona de urgência. Picos altos em 20-30 dias é saudável (tempo pra planejar).

In [ ]:
hoje = pd.Timestamp.now().normalize()

ativas = demandas[
    (demandas["status"] != "CONCLUIDO")
    & (demandas["prazo"].notna())
].copy()
ativas["dias_ate_prazo"] = (ativas["prazo"] - hoje).dt.days

# Remove outliers extremos pra não achatar o histograma
ativas_plot = ativas[ativas["dias_ate_prazo"].between(-30, 90)]

fig = px.histogram(
    ativas_plot,
    x="dias_ate_prazo",
    nbins=40,
    title="Distribuição de dias até o prazo (demandas ativas)",
    labels={"dias_ate_prazo": "Dias até vencer", "count": "Quantidade"},
    color_discrete_sequence=["#3b82f6"],
)
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Hoje", annotation_position="top")
fig.show()

# Resumo
vencidas = (ativas["dias_ate_prazo"] < 0).sum()
urgentes = ((ativas["dias_ate_prazo"] >= 0) & (ativas["dias_ate_prazo"] <= 3)).sum()
print(f"⚠  {vencidas} demandas ativas já ultrapassaram o prazo")
print(f"🔥 {urgentes} demandas vencem nos próximos 3 dias")

## 6. KPI 4 — Top 15 atos mais frequentes

Quais tipos de ato dominam a rotina. Serve pra entender onde vale a pena investir em automação ou template.

> **Pra ler**: os atos do topo são candidatos naturais a macro/template. Se "Ciência de decisão" é 30% do volume, automação dessa leitura tem alto ROI.

In [ ]:
top_atos = (
    demandas["ato"]
    .fillna("(sem ato)")
    .value_counts()
    .head(15)
    .reset_index()
)
top_atos.columns = ["ato", "qtd"]

fig = px.bar(
    top_atos,
    x="qtd",
    y="ato",
    orientation="h",
    title="Top 15 atos mais frequentes",
    labels={"qtd": "Quantidade", "ato": "Ato"},
    color="qtd",
    color_continuous_scale="Blues",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig.show()

## 7. KPI 5 — Carga por defensor e atribuição

Quem tem quantas demandas ativas, distribuídas por que tipo. Mostra desequilíbrios de carga.

> **Pra ler**: defensor com barra muito mais alta que a média pode estar sobrecarregado. Defensor acumulando atribuições que não são dele pode indicar problema na distribuição automática.

In [ ]:
# Exclui 'CONCLUIDO' pra medir só carga ativa
ativas_carga = demandas[demandas["status"] != "CONCLUIDO"]

carga = (
    ativas_carga.groupby(["defensor_nome", "atribuicao_label"])
    .size()
    .reset_index(name="total")
)

# Ordena defensores pelo total
ordem_def = (
    carga.groupby("defensor_nome")["total"].sum().sort_values(ascending=False).index.tolist()
)

fig = px.bar(
    carga,
    x="defensor_nome",
    y="total",
    color="atribuicao_label",
    title="Carga ativa por defensor (excluindo concluídas)",
    labels={"defensor_nome": "Defensor", "total": "Demandas ativas", "atribuicao_label": "Atribuição"},
    barmode="stack",
    category_orders={"defensor_nome": ordem_def},
)
fig.update_xaxes(tickangle=30)
fig.show()

## 8. Bônus — Réu preso com prazo urgente

Recorte operacional crítico: demandas de réu preso com prazo nos próximos 5 dias. São as que **nunca** podem ser esquecidas.

In [ ]:
presos_urgentes = demandas[
    (demandas["reu_preso"] == True)
    & (demandas["status"] != "CONCLUIDO")
    & (demandas["prazo"].notna())
].copy()
presos_urgentes["dias_ate_prazo"] = (presos_urgentes["prazo"] - hoje).dt.days
presos_urgentes = presos_urgentes[presos_urgentes["dias_ate_prazo"] <= 5].sort_values("dias_ate_prazo")

colunas = [
    "dias_ate_prazo",
    "ato",
    "status",
    "atribuicao_label",
    "assistido_nome",
    "defensor_nome",
    "numero_autos",
]
print(f"⚠ {len(presos_urgentes)} demandas de réu preso com prazo <= 5 dias\n")
presos_urgentes[colunas].head(20)

## Conclusão e próximos passos

Se os 5 KPIs acima (+ o recorte bônus) fazem sentido operacional, eles viram **views materializadas** na Fase 1 do TDD. Cada KPI deste notebook corresponde a uma view SQL que o tRPC vai consumir e renderizar como card no dashboard admin.

**Checklist de validação**:

- [ ] Throughput semanal tem a forma esperada (picos, vales, tendência)
- [ ] Backlog por atribuição × status ajuda a identificar gargalo
- [ ] Distribuição de dias até prazo destaca demandas críticas corretamente
- [ ] Top atos captura bem a rotina real
- [ ] Carga por defensor reflete a distribuição real
- [ ] Réu preso urgente retorna os casos que você realmente acompanha

Cada item marcado = uma view SQL para criar. Cada item NÃO marcado = revisar a métrica antes de promover.

**Next**: `notebooks/02-linkage-drive.ipynb` — indexação Drive via path (Estratégia A do TDD).